<a href="https://colab.research.google.com/github/SanjaySaatyaki/hf_llm_course/blob/main/unit_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

In [ ]:
sequences = ["I Like ice-cream","I execerise daily"]

In [ ]:
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

In [ ]:
batch["labels"] = torch.tensor([1,1])

In [ ]:
optmizer = AdamW(model.parameters())
loss = model(**batch).loss
loss.backward()
optmizer.step()

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

In [ ]:
raw_datasets["train"].features

In [ ]:
raw_train_dataset = raw_datasets["train"]

In [ ]:
raw_datasets["train"][15], raw_datasets["validation"][87]

In [ ]:
def tokenization_function(example):
  return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

In [ ]:
tokenized_datasets = raw_datasets.map(tokenization_function,batched=True)


In [ ]:
tokenized_datasets

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Practice

In [ ]:
from datasets import load_dataset

raw_datasets_sst2 = load_dataset("glue","sst2")

In [ ]:
raw_datasets_sst2

In [ ]:
def tokenizer_func(example):
  keys = example.keys()
  values_to_remove = ['label', 'idx']
  keys_to_tokenize = list(filter(lambda item: item not in values_to_remove, keys))
  if len(keys_to_tokenize) == 1:
    return tokenizer(example[keys_to_tokenize[0]], truncation=True)
  # Extract the sentences based on the identified keys
  elif len(keys_to_tokenize) == 2:
    return tokenizer(example[keys_to_tokenize[0]],example[keys_to_tokenize[1]], truncation=True )

  # Pass them to the tokenizer


In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

In [ ]:
tokenizer_func(raw_datasets_sst2['train'][0])

In [ ]:
raw_datasets_sst2.map(tokenizer_func, batched=True)

In [ ]:
raw_datasets_qnli = load_dataset("glue","qnli")
raw_datasets_qnli.map(tokenizer_func)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer",report_to="none")

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["validation"],
    data_collator= data_collator,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

In [ ]:
import numpy as np
preds = np.argmax(predictions.predictions, axis=-1)

In [ ]:
preds[:5]

In [ ]:
# !pip install evaluate

In [ ]:
import evaluate
metric = evaluate.load("glue","mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

In [ ]:
def compute_metrics(eval_preds):
  metric = evaluate.load("glue","mrpc")
  logits, labels = eval_preds
  predictions = np.argmax(logits, axis=-1)
  return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments("test-trainer", eval_strategy="epoch",report_to="none")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

A full Training Loop

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue","mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenizer_function(example):
  return tokenizer(example["sentence1"],example["sentence2"],truncation=True)

tokenized_datasets = raw_datasets.map(tokenizer_function,batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1","sentence2","idx"])
tokenized_datasets = tokenized_datasets.rename_column("label","labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names


In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator)
eval_dataloader = DataLoader(tokenized_datasets["validation"],batch_size=8,collate_fn=data_collator)

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

In [ ]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)